In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../../data/A0008D - v_pt_ride_all (full).csv')

In [3]:
df['BIZ_DT'] = pd.to_datetime(df['BIZ_DT'])
df['ENTRY_DT'] = pd.to_datetime(df['ENTRY_DT'])
df['EXIT_DT'] = pd.to_datetime(df['EXIT_DT'])
df['ENTRY_TM'] = pd.to_datetime(
    df['ENTRY_DT'].astype(str) + ' ' + df['ENTRY_TM'],
    format='%Y-%m-%d %H:%M:%S.%f'
)
df['EXIT_TM'] = pd.to_datetime(
    df['EXIT_DT'].astype(str) + ' ' + df['EXIT_TM'],
    format='%Y-%m-%d %H:%M:%S.%f'
)

In [4]:
df.dtypes

BIZ_DT                datetime64[us]
BUS_SVC_NUM                  float64
CRD_NUM                        int64
DEST_LOC_ID_NUM                int64
ENTRY_DT              datetime64[us]
ENTRY_TM              datetime64[us]
EXIT_DT               datetime64[us]
EXIT_TM               datetime64[us]
JRNY_ID_NUM                    int64
ORIG_LOC_ID_NUM                int64
PATRON_CATG_ID_NUM             int64
PAY_CD                         int64
RIDE_DISC_AMT                float64
RIDE_DIST_KM_CNT             float64
RIDE_FARE_AMT                float64
RIDE_ID_NUM                    int64
RIDE_MIN_CNT                 float64
RIDE_ST_CD                     int64
SVC_GRADE_ID_NUM               int64
TKT_TYP_CD                     int64
TRNSPT_MODE_CD                 int64
dtype: object

In [5]:
df1 = df[df['ENTRY_DT'] == '2025-02-11'].copy()
df1.columns
df2 = df1[['BUS_SVC_NUM', 'CRD_NUM', 'DEST_LOC_ID_NUM', 'ENTRY_DT',
       'ENTRY_TM', 'EXIT_DT', 'EXIT_TM', 'ORIG_LOC_ID_NUM', 'RIDE_DISC_AMT', 'RIDE_DIST_KM_CNT',
       'RIDE_FARE_AMT', 'RIDE_ID_NUM', 'RIDE_MIN_CNT']]
df2.head()

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,RIDE_DIST_KM_CNT,RIDE_FARE_AMT,RIDE_ID_NUM,RIDE_MIN_CNT
58167,62.0,200003194601,6492,2025-02-11,2025-02-11 00:47:26,2025-02-11,2025-02-11 01:13:17,3979,0.0,9.1,1.73,110848327669,25.850
59621,117.0,230021568683,6034,2025-02-11,2025-02-11 00:23:54,2025-02-11,2025-02-11 00:29:20,6042,0.0,1.1,1.19,110848360426,5.433
60686,NaN,230036729632,325,2025-02-11,2025-02-11 00:03:46,2025-02-11,2025-02-11 00:18:29,320,0.0,6.1,1.50,110848362684,14.717
61474,235.0,2190006112681,3380,2025-02-11,2025-02-11 00:33:00,2025-02-11,2025-02-11 00:36:29,6984,0.0,0.8,0.00,110848329634,3.483
61528,979.0,190002646194,2997,2025-02-11,2025-02-11 00:01:31,2025-02-11,2025-02-11 00:29:25,6901,0.0,6.7,0.24,110848330026,27.900


In [6]:
bus_data = pd.read_csv('../../data/A0008D - v_q_bus_stop (full).csv')
bus_vls_data = pd.read_csv('../../data/A0008D - v_q_vls_marker (full).csv')

In [7]:
bus_consolidated = bus_data.merge(bus_vls_data, how= 'left', left_on= 'MRK_ID_NUM',
                                  right_on='MRK_ID_NUM')
bus_needed = bus_consolidated[['BUS_STOP_NAM', 'MRK_ID_NUM', 'LATITUDE_VAL', 'LONGITUDE_VAL']].copy()
bus_needed = bus_needed.rename(columns={
    'LATITUDE_VAL': 'LATITUDE',
    'LONGITUDE_VAL': 'LONGITUDE',
    'BUS_STOP_NAM': 'STATION_NAME'
})
bus_needed['Travel_Type'] = 'BUS'
bus_needed['LATITUDE'] = bus_needed['LATITUDE'] / 3600000
bus_needed['LONGITUDE'] = bus_needed['LONGITUDE'] / 3600000
bus_needed.head(5)

,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
0,Opp Mandai Rd,2940,1.411089,103.755623,BUS
1,SAF FERRY TERMINAL,6367,1.388041,103.999404,BUS
2,Loyang Court,6981,0.000000,0.000000,BUS
3,Os Jurong Frog Farm,6251,0.000000,0.000000,BUS
4,JURONG DEPOT,1771,1.324372,103.736655,BUS


In [8]:
train_data = pd.read_csv('../../data/mrt_station_coordinates.csv')
train_needed = train_data[['NODE_NAME', 'NODE_MRK_ID', 'LATITUDE', 'LONGITUDE']]
train_needed = train_needed.rename(columns={
    'NODE_NAME': 'STATION_NAME',
    'NODE_MRK_ID': 'MRK_ID_NUM'
})
train_needed['Travel_Type'] = 'TRAIN'
train_needed.head(5)

,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
0,Yio Chu Kang,1,1.381756,103.844947,TRAIN
1,Ang Mo Kio,2,1.369933,103.849558,TRAIN
2,Bishan NSEW,3,1.350839,103.848144,TRAIN
3,Braddell,4,1.340469,103.846799,TRAIN
4,Toa Payoh,5,1.332629,103.847502,TRAIN


In [9]:
consolidated_stations = pd.concat([bus_needed, train_needed], axis = 0)
consolidated_stations.tail(5)

,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
201,Marine Parade,427,1.302865,103.905508,TRAIN
202,Marine Terrace,428,1.306786,103.915316,TRAIN
203,Siglap,429,1.309877,103.929879,TRAIN
204,Bayshore,430,1.312841,103.941597,TRAIN
205,Hume,336,1.354511,103.769104,TRAIN


In [10]:
df2 = df2.merge(consolidated_stations, how = 'left', left_on= 'DEST_LOC_ID_NUM',
                right_on= 'MRK_ID_NUM')
df2.head(5)

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,RIDE_DIST_KM_CNT,RIDE_FARE_AMT,RIDE_ID_NUM,RIDE_MIN_CNT,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
0,62.0,200003194601,6492,2025-02-11,2025-02-11 00:47:26,2025-02-11,2025-02-11 01:13:17,3979,0.0,9.1,1.73,110848327669,25.850,Coral Edge Stn Exit A,6492.0,1.394307,103.911955,BUS
1,117.0,230021568683,6034,2025-02-11,2025-02-11 00:23:54,2025-02-11,2025-02-11 00:29:20,6042,0.0,1.1,1.19,110848360426,5.433,Opp Blk 104B,6034.0,1.447964,103.830401,BUS
2,NaN,230036729632,325,2025-02-11,2025-02-11 00:03:46,2025-02-11,2025-02-11 00:18:29,320,0.0,6.1,1.50,110848362684,14.717,Ubi,325.0,1.329974,103.899227,TRAIN
3,235.0,2190006112681,3380,2025-02-11,2025-02-11 00:33:00,2025-02-11,2025-02-11 00:36:29,6984,0.0,0.8,0.00,110848329634,3.483,Braddell Stn/Blk 111,3380.0,1.340863,103.845456,BUS
4,979.0,190002646194,2997,2025-02-11,2025-02-11 00:01:31,2025-02-11,2025-02-11 00:29:25,6901,0.0,6.7,0.24,110848330026,27.900,Regent Gr Condo,2997.0,1.399841,103.749913,BUS


In [11]:
df2 = df2.rename(columns={
    'STATION_NAME': 'DEST_STATION_NAME',
    'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
    'LATITUDE': 'DEST_LATITUDE',
    'LONGITUDE': 'DEST_LONGITUDE',
    'Travel_Type': 'DEST_Travel_Type'
})

In [12]:
df2 = df2.merge(consolidated_stations, how = 'left', left_on= 'ORIG_LOC_ID_NUM',
                right_on= 'MRK_ID_NUM')

In [13]:
df2 = df2.rename(columns={
    'STATION_NAME': 'ORIG_STATION_NAME',
    'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
    'LATITUDE': 'ORIG_LATITUDE',
    'LONGITUDE': 'ORIG_LONGITUDE',
    'Travel_Type': 'ORIG_Travel_Type'
})
df2.head(5)

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,RIDE_DIST_KM_CNT,...,DEST_STATION_NAME,DEST_MRK_ID_NUM,DEST_LATITUDE,DEST_LONGITUDE,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type
0,62.0,200003194601,6492,2025-02-11,2025-02-11 00:47:26,2025-02-11,2025-02-11 01:13:17,3979,0.0,9.1,...,Coral Edge Stn Exit A,6492.0,1.394307,103.911955,BUS,The Minton,3979.0,1.351073,103.882180,BUS
1,117.0,230021568683,6034,2025-02-11,2025-02-11 00:23:54,2025-02-11,2025-02-11 00:29:20,6042,0.0,1.1,...,Opp Blk 104B,6034.0,1.447964,103.830401,BUS,Canberra Stn,6042.0,1.442578,103.830142,BUS
2,NaN,230036729632,325,2025-02-11,2025-02-11 00:03:46,2025-02-11,2025-02-11 00:18:29,320,0.0,6.1,...,Ubi,325.0,1.329974,103.899227,TRAIN,Jalan Besar,320.0,1.305171,103.855296,TRAIN
3,235.0,2190006112681,3380,2025-02-11,2025-02-11 00:33:00,2025-02-11,2025-02-11 00:36:29,6984,0.0,0.8,...,Braddell Stn/Blk 111,3380.0,1.340863,103.845456,BUS,Caldecott Stn Exit 1,6984.0,1.336502,103.839633,BUS
4,979.0,190002646194,2997,2025-02-11,2025-02-11 00:01:31,2025-02-11,2025-02-11 00:29:25,6901,0.0,6.7,...,Regent Gr Condo,2997.0,1.399841,103.749913,BUS,Bukit Panjang Interchange B3,6901.0,1.378685,103.762778,BUS
